# 04_mop_data_cleaning.ipynb
## Project Title: Traffic Accident Risk Prediction (TARP)
**Unit:** SIT782  
**Prepared by:** Subathira Thinakaran  

**Project Team:**  
- Suba (225094537)  
- Burhan (224802775)  
- Khalid (224696667)  

**Task:** Clean MOP Pedestrian and Bicycle Data (Week 2 of 8)

## Objective
This notebook cleans the pedestrian and bicycle datasets collected from the City of Melbourne Open Data API. The cleaning process includes handling missing values, standardising formats, preparing time-related fields, and aggregating mobility counts by hour for later integration and analysis.

## Datasets Used
/data/raw/pedestrian.csv
/data/raw/bicycle.csv

## Output
/data/processed/bicycle_clean.csv
/data/processed/bicycle_long.csv
/data/processed/bicycle_hourly.csv
/data/processed/pedestrian_clean.csv



## Imports and Setup

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import ast

In [3]:
from google.colab import files
uploaded = files.upload()

Saving bicycle.csv to bicycle.csv
Saving pedestrian.csv to pedestrian.csv


## Load Raw Datasets

Load the pedestrian and bicycle datasets collected from the previous data collection stage.  
This step verifies that the datasets are available and ready for cleaning.

In [33]:
# -----------------------------
# 1. Load raw datasets
# -----------------------------
from pathlib import Path
import pandas as pd

PROCESSED_DATA_DIR = Path("data/processed")
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

pedestrian_file = Path("pedestrian.csv")
bicycle_file = Path("bicycle.csv")

pedestrian_df = pd.read_csv(pedestrian_file)
bicycle_df = pd.read_csv(bicycle_file)

print("Pedestrian raw shape:", pedestrian_df.shape)
print("Bicycle raw shape:", bicycle_df.shape)

Pedestrian raw shape: (5000, 9)
Bicycle raw shape: (5000, 28)


### Observation

Both raw datasets were loaded successfully. The pedestrian dataset has a compact structure, while the bicycle dataset contains a richer set of traffic-related attributes that will require more selective cleaning.

## Initial Inspection

Inspect dataset structure and identify data quality issues such as missing values and unnecessary features.

In [35]:
# -----------------------------
# 2. Initial inspection
# -----------------------------
print("Pedestrian columns:")
print(list(pedestrian_df.columns))

print("\nBicycle columns:")
print(list(bicycle_df.columns))

print("\nPedestrian missing values:")
print(pedestrian_df.isna().sum())

print("\nTop bicycle missing values:")
missing_bicycle = bicycle_df.isna().sum().sort_values(ascending=False)
print(missing_bicycle.head(15))

# Focus on key analysis column
missing_bike = bicycle_df["bike"].isna().sum()
print(f"\nMissing 'bike' values: {missing_bike}")
print(f"Available 'bike' records: {len(bicycle_df) - missing_bike}")

Pedestrian columns:
['id', 'location_id', 'sensing_date', 'hourday', 'direction_1', 'direction_2', 'pedestriancount', 'sensor_name', 'location']

Bicycle columns:
['date', 'road_name', 'location', 'suburb', 'speed_limit', 'direction', 'time', 'vehicle_class_1', 'vehicle_class_2', 'vehicle_class_3', 'vehicle_class_4', 'vehicle_class_5', 'vehicle_class_6', 'vehicle_class_7', 'vehicle_class_8', 'vehicle_class_9', 'vehicle_class_10', 'vehicle_class_11', 'vehicle_class_12', 'vehicle_class_13', 'motorcycle', 'bike', 'average_speed', '85th_percentile_speed', 'maximum_speed', 'road_segment', 'road_segment_1', 'road_segment_2']

Pedestrian missing values:
id                 0
location_id        0
sensing_date       0
hourday            0
direction_1        0
direction_2        0
pedestriancount    0
sensor_name        0
location           0
dtype: int64

Top bicycle missing values:
road_segment_2           4765
road_segment_1           4592
motorcycle               1891
bike                    

### Observation

The pedestrian dataset does not contain any missing values and is already well-structured for analysis.

In contrast, the bicycle dataset contains missing values across several columns. The most important issue is the `bike` column, which has a significant number of missing values, indicating that bicycle counts were not recorded in all traffic surveys.

Other columns such as `road_segment_1`, `road_segment_2`, and several vehicle classification fields also contain high levels of missing data and may not be relevant for this project.

These findings indicate that the bicycle dataset will require selective cleaning, including filtering valid bike records and removing irrelevant or sparsely populated columns.

## Clean Pedestrian Dataset

The pedestrian dataset requires relatively light cleaning, including standardising columns, converting dates, extracting coordinates, and creating a timestamp field.

In [37]:
# -----------------------------
# 3. Clean pedestrian dataset
# -----------------------------
pedestrian_clean = pedestrian_df.copy()

# Standardise column names
pedestrian_clean.columns = pedestrian_clean.columns.str.strip().str.lower()

# Convert date and hour
pedestrian_clean["sensing_date"] = pd.to_datetime(
    pedestrian_clean["sensing_date"], errors="coerce"
)
pedestrian_clean["hourday"] = pd.to_numeric(
    pedestrian_clean["hourday"], errors="coerce"
)

# Check duplicates
print("Duplicate pedestrian rows:", pedestrian_clean.duplicated().sum())

# Check missing values after conversion
print("\nMissing values after conversion:")
print(pedestrian_clean[["sensing_date", "hourday"]].isna().sum())

print("\nPedestrian data types:")
print(pedestrian_clean.dtypes)

Duplicate pedestrian rows: 0

Missing values after conversion:
sensing_date    0
hourday         0
dtype: int64

Pedestrian data types:
id                          int64
location_id                 int64
sensing_date       datetime64[ns]
hourday                     int64
direction_1                 int64
direction_2                 int64
pedestriancount             int64
sensor_name                object
location                   object
dtype: object


### Observation

The pedestrian dataset required minimal cleaning. Column names were standardised, and the `sensing_date` and `hourday` fields were successfully converted into appropriate data types.

No duplicate records or missing values were identified after conversion, indicating that the dataset is clean and reliable for further processing.

### Extract Geographic Coordinates

The location field contains nested longitude and latitude values.  
This step extracts them into separate columns for easier analysis.

In [39]:
# -----------------------------
# 3A. Extract location coordinates
# -----------------------------
def extract_coords(x):
    try:
        if pd.isna(x):
            return pd.Series([None, None])

        # Convert string to dictionary if needed
        loc = ast.literal_eval(x) if isinstance(x, str) else x

        # Extract coordinates safely
        lat = loc.get("lat") if isinstance(loc, dict) else None
        lon = loc.get("lon") if isinstance(loc, dict) else None

        return pd.Series([lat, lon])

    except Exception:
        return pd.Series([None, None])

# Apply extraction
pedestrian_clean[["latitude", "longitude"]] = pedestrian_clean["location"].apply(extract_coords)

print("Missing latitude:", pedestrian_clean["latitude"].isna().sum())
print("Missing longitude:", pedestrian_clean["longitude"].isna().sum())

Missing latitude: 0
Missing longitude: 0


### Observation

Latitude and longitude values were successfully extracted from the `location` field. No missing coordinate values were observed, indicating that all records contain valid spatial information. This enables spatial analysis of pedestrian activity in later stages of the project.

### Finalise Pedestrian Dataset

Remove redundant columns and retain only useful features for analysis.

In [42]:
# -----------------------------
# 3B. Finalise pedestrian dataset
# -----------------------------
# Drop location column only if it exists
if "location" in pedestrian_clean.columns:
    pedestrian_clean = pedestrian_clean.drop(columns=["location"])

# Create full timestamp
pedestrian_clean["timestamp"] = (
    pedestrian_clean["sensing_date"] +
    pd.to_timedelta(pedestrian_clean["hourday"], unit="h")
)

# Reorder columns
pedestrian_clean = pedestrian_clean[
    [
        "id", "location_id", "sensor_name",
        "sensing_date", "hourday", "timestamp",
        "direction_1", "direction_2", "pedestriancount",
        "latitude", "longitude"
    ]
]

print("Final pedestrian shape:", pedestrian_clean.shape)
pedestrian_clean.head()

Final pedestrian shape: (5000, 11)


,id,location_id,sensor_name,sensing_date,hourday,timestamp,direction_1,direction_2,pedestriancount,latitude,longitude
0,135920240721,135,Spen161_T,2024-07-21,9,2024-07-21 09:00:00,226,225,451,-37.817286,144.953191
1,135220250609,135,Spen161_T,2025-06-09,2,2025-06-09 02:00:00,9,21,30,-37.817286,144.953191
2,132520250803,132,King335_T,2025-08-03,5,2025-08-03 05:00:00,8,4,12,-37.812676,144.953864
3,1342020240829,134,Spen201_T,2024-08-29,20,2024-08-29 20:00:00,311,233,544,-37.816942,144.953039
4,1342020260104,134,Spen201_T,2026-01-04,20,2026-01-04 20:00:00,264,253,517,-37.816942,144.953039


### Observation

The pedestrian dataset was finalised by removing the original nested location field, creating a full timestamp, and arranging the columns into a cleaner structure. The resulting dataset is well prepared for later exploratory analysis and feature engineering.

## Clean Bicycle Dataset
Remove fully empty columns and retain the most relevant mobility, spatial, and time-based features for later analysis.

In [44]:
# -----------------------------
# 4. Clean bicycle dataset
# -----------------------------
bicycle_clean = bicycle_df.copy()
bicycle_clean.columns = bicycle_clean.columns.str.strip().str.lower()

print("Original bicycle shape:", bicycle_clean.shape)
print("Duplicate bicycle rows:", bicycle_clean.duplicated().sum())

print("\nKey columns available:")
print(["date", "time", "bike", "road_name", "location", "suburb", "direction"])

Original bicycle shape: (5000, 28)
Duplicate bicycle rows: 0

Key columns available:
['date', 'time', 'bike', 'road_name', 'location', 'suburb', 'direction']


### Observation

The bicycle dataset contains no duplicate rows, which simplifies the cleaning process. The next steps will focus on converting date and time fields, handling missing bike counts, and selecting the most relevant columns for mobility analysis.

## Convert Core Fields

This step converts key columns in the bicycle dataset into appropriate data types to enable accurate analysis.

The `date` column is converted into a datetime format, while the `bike` and speed-related fields are converted into numeric values. This ensures that these fields can be used for aggregation, filtering, and statistical analysis.

Additionally, missing values in critical fields such as `bike` are identified, as they will need to be handled in the next step of the cleaning process.

In [46]:
# -----------------------------
# 4A. Convert core fields
# -----------------------------
bicycle_clean["date"] = pd.to_datetime(bicycle_clean["date"], errors="coerce")

numeric_columns = [
    "bike",
    "speed_limit",
    "average_speed",
    "85th_percentile_speed",
    "maximum_speed"
]

for col in numeric_columns:
    bicycle_clean[col] = pd.to_numeric(bicycle_clean[col], errors="coerce")

print("Missing date values:", bicycle_clean["date"].isna().sum())
print("Missing bike values:", bicycle_clean["bike"].isna().sum())

Missing date values: 0
Missing bike values: 1891


### Observation

The `date` column was successfully converted into datetime format with no missing values. However, the `bike` column contains a significant number of missing values, indicating that bicycle counts were not recorded in all observations.

This confirms the need to filter or handle missing bike records in the next step to ensure accurate analysis.

## Filter Valid Bicycle Records

This step removes records where bicycle counts are missing. Since the `bike` column is the main variable required for bicycle mobility analysis, only rows with valid bicycle count values are retained for further processing.

In [48]:
# -----------------------------
# 4B. Keep records with valid bike counts
# -----------------------------
initial_rows = len(bicycle_clean)

bicycle_clean = bicycle_clean.dropna(subset=["bike"]).copy()

print("Rows before filtering:", initial_rows)
print("Rows after filtering valid bike counts:", len(bicycle_clean))
print("Rows removed:", initial_rows - len(bicycle_clean))

bicycle_clean.head()

Rows before filtering: 3109
Rows after filtering valid bike counts: 3109
Rows removed: 0


,date,road_name,location,suburb,speed_limit,direction,time,vehicle_class_1,vehicle_class_2,vehicle_class_3,...,vehicle_class_12,vehicle_class_13,motorcycle,bike,average_speed,85th_percentile_speed,maximum_speed,road_segment,road_segment_1,road_segment_2
37,2015-04-24,Rosslyn Street,Between Adderley Street and Spencer Street,West Melbourne,50,E,6:00,6.0,0.0,1.0,...,0.0,0.0,0.0,1.0,26.0,29.0,35.0,21479,NaN,NaN
38,2015-04-24,Rosslyn Street,Between Adderley Street and Spencer Street,West Melbourne,50,E,16:00,38.0,0.0,1.0,...,0.0,0.0,0.0,0.0,31.0,39.0,51.0,21479,NaN,NaN
39,2015-04-24,Rosslyn Street,Between Adderley Street and Spencer Street,West Melbourne,50,E,19:00,10.0,0.0,1.0,...,0.0,0.0,0.0,1.0,32.0,36.0,43.0,21479,NaN,NaN
40,2015-04-25,Rosslyn Street,Between Adderley Street and Spencer Street,West Melbourne,50,E,15:00,6.0,0.0,0.0,...,0.0,0.0,0.0,0.0,20.0,23.0,25.0,21479,NaN,NaN
41,2015-04-25,Rosslyn Street,Between Adderley Street and Spencer Street,West Melbourne,50,E,18:00,4.0,0.0,0.0,...,0.0,0.0,1.0,0.0,44.0,48.0,64.0,21479,NaN,NaN


### Observation

After filtering out records with missing bicycle counts, 3109 valid rows remain for analysis. This indicates that a substantial portion of the dataset contains usable bicycle traffic information, while rows without recorded bike counts have been removed to improve the reliability of subsequent analysis.

## Create Time-Based Features

This step derives additional temporal features from the bicycle dataset to support time-based analysis.

The `hour` field is extracted from the `time` column, while `day_of_week` and `is_weekend` are derived from the `date` field. These features will enable hourly, daily, and weekday-based analysis in the next stage.

In [50]:
# -----------------------------
# 4C. Create time-based features
# -----------------------------
# Extract hour from time column
bicycle_clean["hour"] = pd.to_datetime(
    bicycle_clean["time"], format="%H:%M", errors="coerce"
).dt.hour

# Create day-based features
bicycle_clean["day_of_week"] = bicycle_clean["date"].dt.day_name()
bicycle_clean["is_weekend"] = bicycle_clean["day_of_week"].isin(["Saturday", "Sunday"])

# Validate results
print("Missing hour values:", bicycle_clean["hour"].isna().sum())
print("Unique days:", bicycle_clean["day_of_week"].unique())

bicycle_clean[["date", "time", "hour", "day_of_week", "is_weekend"]].head()

Missing hour values: 0
Unique days: ['Friday' 'Saturday' 'Sunday' 'Monday' 'Tuesday' 'Thursday' 'Wednesday']


,date,time,hour,day_of_week,is_weekend
37,2015-04-24,6:00,6,Friday,False
38,2015-04-24,16:00,16,Friday,False
39,2015-04-24,19:00,19,Friday,False
40,2015-04-25,15:00,15,Saturday,True
41,2015-04-25,18:00,18,Saturday,True


### Observation

The time-based features were successfully created with no missing values. The dataset now includes hour-level and day-level attributes, allowing for detailed temporal analysis of bicycle traffic patterns, including weekday and weekend comparisons.

## Select Relevant Bicycle Features

This step reduces the bicycle dataset to only the most relevant columns required for analysis.

Irrelevant or sparsely populated columns, such as detailed vehicle classifications and largely missing road segment fields, are excluded to simplify the dataset and improve data quality for subsequent analysis.

In [52]:
# -----------------------------
# 4D. Select relevant bicycle columns
# -----------------------------
selected_bicycle_columns = [
    "date", "day_of_week", "is_weekend", "time", "hour",
    "road_name", "location", "suburb", "direction",
    "speed_limit", "bike", "average_speed",
    "85th_percentile_speed", "maximum_speed",
    "road_segment"   # keep only main segment
]

selected_bicycle_columns = [
    col for col in selected_bicycle_columns if col in bicycle_clean.columns
]

bicycle_clean = bicycle_clean[selected_bicycle_columns]

print("Final bicycle shape:", bicycle_clean.shape)
bicycle_clean.head()

Final bicycle shape: (3109, 15)


,date,day_of_week,is_weekend,time,hour,road_name,location,suburb,direction,speed_limit,bike,average_speed,85th_percentile_speed,maximum_speed,road_segment
37,2015-04-24,Friday,False,6:00,6,Rosslyn Street,Between Adderley Street and Spencer Street,West Melbourne,E,50,1.0,26.0,29.0,35.0,21479
38,2015-04-24,Friday,False,16:00,16,Rosslyn Street,Between Adderley Street and Spencer Street,West Melbourne,E,50,0.0,31.0,39.0,51.0,21479
39,2015-04-24,Friday,False,19:00,19,Rosslyn Street,Between Adderley Street and Spencer Street,West Melbourne,E,50,1.0,32.0,36.0,43.0,21479
40,2015-04-25,Saturday,True,15:00,15,Rosslyn Street,Between Adderley Street and Spencer Street,West Melbourne,E,50,0.0,20.0,23.0,25.0,21479
41,2015-04-25,Saturday,True,18:00,18,Rosslyn Street,Between Adderley Street and Spencer Street,West Melbourne,E,50,0.0,44.0,48.0,64.0,21479


### Observation

The bicycle dataset has been reduced to a focused set of relevant features, including temporal attributes, location details, bicycle counts, and selected speed-related variables.

Highly sparse or less relevant columns have been removed to improve data quality and simplify further analysis. The resulting dataset is well-structured and suitable for exploratory analysis and feature engineering.

## Bicycle Hourly Summary

This step aggregates bicycle counts by hour to analyse how bicycle traffic varies throughout the day.

In [54]:
# -----------------------------
# 5. Create bicycle hourly summary
# -----------------------------
bicycle_hourly = (
    bicycle_clean
    .groupby("hour", as_index=False)["bike"]
    .sum()
    .sort_values("hour")
    .reset_index(drop=True)
)

print("Bicycle hourly shape:", bicycle_hourly.shape)

print("\nHour range:", bicycle_hourly["hour"].min(), "-", bicycle_hourly["hour"].max())

bicycle_hourly.head()

Bicycle hourly shape: (24, 2)

Hour range: 0 - 23


,hour,bike
0,0,47.0
1,1,57.0
2,2,22.0
3,3,47.0
4,4,37.0


### Observation

The hourly aggregation shows that bicycle traffic is distributed across all 24 hours of the day. This enables detailed analysis of peak and low activity periods, which is important for understanding mobility patterns and potential accident risk periods.

## Bicycle Daily Summary

This step aggregates bicycle counts by date to analyse how bicycle traffic varies across different days.

In [56]:
# -----------------------------
# 5A. Create bicycle daily summary
# -----------------------------
bicycle_daily = (
    bicycle_clean
    .groupby("date", as_index=False)["bike"]
    .sum()
    .sort_values("date")
    .reset_index(drop=True)
)

print("Bicycle daily shape:", bicycle_daily.shape)

print("\nDate range:",
      bicycle_daily["date"].min(),
      "to",
      bicycle_daily["date"].max())

bicycle_daily.head()

Bicycle daily shape: (249, 2)

Date range: 2015-01-03 00:00:00 to 2017-10-02 00:00:00


,date,bike
0,2015-01-03,9.0
1,2015-01-06,6.0
2,2015-01-09,90.0
3,2015-01-12,32.0
4,2015-02-03,17.0


### Observation

The daily aggregation shows bicycle traffic trends across 249 unique dates. This enables analysis of variations in bicycle activity over time, including identifying high and low traffic days, which can be useful for understanding patterns related to accident risk.

## Bicycle Weekday Summary

This step aggregates bicycle counts by day of the week to analyse weekly traffic patterns and identify differences between weekdays and weekends.

In [58]:
# -----------------------------
# 5B. Create bicycle weekday summary
# -----------------------------
weekday_order = [
    "Monday", "Tuesday", "Wednesday",
    "Thursday", "Friday", "Saturday", "Sunday"
]

bicycle_weekday = (
    bicycle_clean
    .groupby("day_of_week", as_index=False)["bike"]
    .sum()
)

bicycle_weekday["day_of_week"] = pd.Categorical(
    bicycle_weekday["day_of_week"],
    categories=weekday_order,
    ordered=True
)

bicycle_weekday = (
    bicycle_weekday
    .sort_values("day_of_week")
    .reset_index(drop=True)
)

print("Bicycle weekday shape:", bicycle_weekday.shape)

print("\nWeekdays covered:", list(bicycle_weekday["day_of_week"]))

bicycle_weekday

Bicycle weekday shape: (7, 2)

Weekdays covered: ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']


,day_of_week,bike
0,Monday,1261.0
1,Tuesday,707.0
2,Wednesday,1094.0
3,Thursday,1195.0
4,Friday,772.0
5,Saturday,895.0
6,Sunday,544.0


### Observation

The weekday aggregation shows variation in bicycle traffic across different days of the week. This enables comparison between weekday and weekend patterns, helping identify periods of higher mobility activity that may be associated with increased accident risk.

## Save Cleaned and Processed Datasets

This section saves the cleaned pedestrian and bicycle datasets, along with derived bicycle summary tables, for later analysis.

In [59]:
# -----------------------------
# 6. Save cleaned datasets
# -----------------------------
pedestrian_clean_file = PROCESSED_DATA_DIR / "pedestrian_clean.csv"
bicycle_clean_file = PROCESSED_DATA_DIR / "bicycle_clean.csv"
bicycle_hourly_file = PROCESSED_DATA_DIR / "bicycle_hourly.csv"
bicycle_daily_file = PROCESSED_DATA_DIR / "bicycle_daily.csv"
bicycle_weekday_file = PROCESSED_DATA_DIR / "bicycle_weekday.csv"

pedestrian_clean.to_csv(pedestrian_clean_file, index=False)
bicycle_clean.to_csv(bicycle_clean_file, index=False)
bicycle_hourly.to_csv(bicycle_hourly_file, index=False)
bicycle_daily.to_csv(bicycle_daily_file, index=False)
bicycle_weekday.to_csv(bicycle_weekday_file, index=False)

print("Saved:", pedestrian_clean_file)
print("Saved:", bicycle_clean_file)
print("Saved:", bicycle_hourly_file)
print("Saved:", bicycle_daily_file)
print("Saved:", bicycle_weekday_file)

Saved: data/processed/pedestrian_clean.csv
Saved: data/processed/bicycle_clean.csv
Saved: data/processed/bicycle_hourly.csv
Saved: data/processed/bicycle_daily.csv
Saved: data/processed/bicycle_weekday.csv


In [62]:
# -----------------------------
# 7. Final summary
# -----------------------------
summary = {
    "Pedestrian cleaned rows": len(pedestrian_clean),
    "Bicycle cleaned rows": len(bicycle_clean),
    "Bicycle hourly rows": len(bicycle_hourly),
    "Bicycle daily rows": len(bicycle_daily),
    "Bicycle weekday rows": len(bicycle_weekday),
    "Pedestrian file": str(pedestrian_clean_file),
    "Bicycle clean file": str(bicycle_clean_file),
    "Bicycle hourly file": str(bicycle_hourly_file),
    "Bicycle daily file": str(bicycle_daily_file),
    "Bicycle weekday file": str(bicycle_weekday_file),
}

print("Final dataset summary:\n")
for key, value in summary.items():
    print(f"{key}: {value}")

Final dataset summary:

Pedestrian cleaned rows: 5000
Bicycle cleaned rows: 3109
Bicycle hourly rows: 24
Bicycle daily rows: 249
Bicycle weekday rows: 7
Pedestrian file: data/processed/pedestrian_clean.csv
Bicycle clean file: data/processed/bicycle_clean.csv
Bicycle hourly file: data/processed/bicycle_hourly.csv
Bicycle daily file: data/processed/bicycle_daily.csv
Bicycle weekday file: data/processed/bicycle_weekday.csv


In [ ]:
# -----------------------------
# 8. Download processed files (optional)
# -----------------------------
from google.colab import files

files.download(str(pedestrian_clean_file))
files.download(str(bicycle_clean_file))
files.download(str(bicycle_hourly_file))
files.download(str(bicycle_daily_file))
files.download(str(bicycle_weekday_file))

## Final Summary

This notebook successfully cleaned and transformed the pedestrian and bicycle datasets obtained from the City of Melbourne Open Data platform.

The pedestrian dataset required minimal preprocessing and was enhanced with spatial coordinates and timestamp information. The bicycle dataset required more extensive cleaning, including handling missing values, filtering valid bicycle counts, converting date and time fields, and creating derived temporal features.

Additional aggregated datasets were created at hourly, daily, and weekday levels to support detailed exploratory data analysis. These processed datasets provide a strong foundation for analysing mobility patterns and developing features for accident risk prediction.